# BIBE RFMI Image-to-Indices Demo

This notebook demonstrates a public, reproducible workflow for converting open-eye frontal facial images into MediaPipe landmark coordinates, coordinate-based overlay images, relative facial morphology indices (RFMI), and summary tables.

The default example uses a synthetic open-eye child facial image. It is not study data and does not depict a real participant.

## 1. Install and Import Required Libraries

Set `INSTALL = True` only if the current Jupyter environment does not already have the required packages.

In [ ]:
from pathlib import Path
import subprocess
import sys

INSTALL = False

# Optional manual override. Use this only if automatic root detection fails.
# Example: ROOT_OVERRIDE = Path(r'C:\\path\\to\\mediapipe_child_face_analysis_RFMI')
ROOT_OVERRIDE = None

def looks_like_project_root(path):
    path = Path(path)
    return (path / 'requirements.txt').exists() and (path / 'scripts' / '01_prepare_project.py').exists()

def find_project_root():
    candidates = []
    if ROOT_OVERRIDE is not None:
        candidates.append(Path(ROOT_OVERRIDE))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    repo_names = ['mediapipe_child_face_analysis_RFMI', 'mediapipe_child_face_analysis_RFMI_repo_ready', 'jupyter_reproducible_workflow']
    for desktop_name in ['Desktop', '桌面']:
        desktop = Path.home() / desktop_name
        for repo_name in repo_names:
            candidates.append(desktop / repo_name)
        if desktop.exists():
            for repo_name in repo_names:
                candidates.extend(desktop.glob(f'**/{repo_name}'))
    for candidate in candidates:
        candidate = candidate.resolve()
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError(
        'Project root was not found. Set ROOT_OVERRIDE to the jupyter_reproducible_workflow folder.'
    )

ROOT = find_project_root()

if INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)

import pandas as pd
from IPython.display import Image, display

print('Project root:', ROOT)

## 2. Prepare Synthetic Example Image and Manifest

The public demo uses one synthetic image. This section prepares the example image and writes a simple manifest. For real studies, replace the synthetic image path with approved participant images under the relevant ethics protocol.

In [ ]:
def run(args):
    cmd = [sys.executable] + [str(a) for a in args]
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

OPEN_IMAGE = ROOT / 'example_data' / 'images' / 'SYN_open.jpg'
SUBJECT_ID = 'SYN'

run([
    ROOT / 'scripts' / '01_prepare_project.py',
    '--root', ROOT,
    '--open-image', OPEN_IMAGE,
    '--child-id', SUBJECT_ID,
])

manifest = pd.read_csv(ROOT / 'outputs_manifest' / 'manifest.csv')
manifest

## 3. Run MediaPipe and Export Raw Landmark Coordinates

This step runs MediaPipe Face Landmarker, exports the raw landmark coordinate table, and records image-level detection information. Overlay points are generated from MediaPipe coordinates; no manual landmark placement is used.

In [ ]:
run([ROOT / 'scripts' / '02_detect_and_overlay.py', '--root', ROOT])

landmarks = pd.read_csv(ROOT / 'outputs_landmarks' / 'landmarks_raw.csv')
detection_log = pd.read_csv(ROOT / 'outputs_landmarks' / 'detection_log.csv')

print('Landmark rows:', len(landmarks))
display(detection_log)
landmarks.head()

## 4. Inspect Coordinate-Based Overlay Images

The workflow generates three overlay types: full-face landmarks, eye-region zoom, and RFMI distance lines.

In [ ]:
image_id = f'{SUBJECT_ID}_open'
overlay_paths = [
    ROOT / 'outputs_overlay' / 'full_face' / f'{image_id}_overlay.jpg',
    ROOT / 'outputs_overlay' / 'eye_zoom' / f'{image_id}_eye_zoom.jpg',
    ROOT / 'outputs_overlay' / 'rfmi_lines' / f'{image_id}_rfmi_lines.jpg',
]

for path in overlay_paths:
    print(path.name)
    display(Image(filename=str(path), width=650))

## 5. Compute RFMI Indices

RFMI values are within-face relative indices. They are not centimeter measurements and do not require camera-distance calibration.

In [ ]:
run([ROOT / 'scripts' / '04_compute_rfmi.py', '--root', ROOT])

rfmi_image = pd.read_csv(ROOT / 'outputs_features' / 'rfmi_image_level.csv')
rfmi_subject = pd.read_csv(ROOT / 'outputs_features' / 'rfmi_subject_level.csv')

display(rfmi_image)
display(rfmi_subject)

## 6. Create RFMI Subject-Level and Summary Tables

The subject-level table has one row per subject and one column per RFMI variable. The summary table reports n, mean, standard deviation, median, minimum, maximum, and mean ± standard deviation for each metric.

In [ ]:
run([ROOT / 'scripts' / '05_summarize_rfmi.py', '--root', ROOT])

subject_table = pd.read_csv(ROOT / 'outputs_stats' / 'tables' / 'rfmi_subject_indices.csv')
summary_table = pd.read_csv(ROOT / 'outputs_stats' / 'tables' / 'rfmi_summary.csv')

display(subject_table)
display(summary_table)

## 7. Main Output Files

The main reproducible outputs are raw coordinates, overlay images, RFMI subject-level indices, and RFMI summary statistics.

In [ ]:
main_outputs = [
    ROOT / 'outputs_manifest' / 'manifest.csv',
    ROOT / 'outputs_landmarks' / 'landmarks_raw.csv',
    ROOT / 'outputs_landmarks' / 'detection_log.csv',
    ROOT / 'outputs_overlay' / 'full_face' / f'{image_id}_overlay.jpg',
    ROOT / 'outputs_overlay' / 'eye_zoom' / f'{image_id}_eye_zoom.jpg',
    ROOT / 'outputs_overlay' / 'rfmi_lines' / f'{image_id}_rfmi_lines.jpg',
    ROOT / 'outputs_features' / 'rfmi_image_level.csv',
    ROOT / 'outputs_features' / 'rfmi_subject_level.csv',
    ROOT / 'outputs_stats' / 'tables' / 'rfmi_subject_indices.csv',
    ROOT / 'outputs_stats' / 'tables' / 'rfmi_summary.csv',
]

for path in main_outputs:
    print(path.relative_to(ROOT), 'exists=', path.exists())